# Valutazione finale sul test su Kaggle

Impostazioni: GPU T4, Internet on. Stessi secret del notebook di training. La valutazione sul test si esegue una sola volta, con le soglie scelte sul validation in `configs/compare_final_e0.yaml`.

In [ ]:
import os
import shutil
import subprocess
from pathlib import Path

from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()


def secret(name, default=None):
    try:
        return secrets.get_secret(name)
    except Exception:
        return default


print(subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True).stdout)
print(shutil.disk_usage("/tmp"))
print(shutil.disk_usage("/kaggle/working"))

In [ ]:
WORK = Path("/kaggle/working/vehicle-detection")
REPO = "github.com/FapaKslapa/vehicle-detection.git"
if not WORK.exists():
    subprocess.run(
        ["git", "clone", "--branch", "development", f"https://{REPO}", str(WORK)],
        check=True,
    )
os.chdir(WORK)
print(subprocess.run(["git", "log", "--oneline", "-3"], capture_output=True, text=True).stdout)


In [ ]:
names = [
    "S3_BUCKET",
    "S3_ENDPOINT_URL",
    "S3_REGION",
    "AWS_ACCESS_KEY_ID",
    "AWS_SECRET_ACCESS_KEY",
    "WANDB_API_KEY",
]
lines = [f"{n}={v}" for n in names if (v := secret(n))]
lines.append("WANDB_MODE=" + ("online" if secret("WANDB_API_KEY") else "disabled"))
Path(".env").write_text("\n".join(lines) + "\n")
os.chmod(".env", 0o600)
print([line.split("=")[0] for line in lines])

In [ ]:
subprocess.run(["pip", "install", "-q", "uv", "rust-just"], check=True)

In [ ]:
env = {
    **os.environ,
    "UV_PROJECT_ENVIRONMENT": "/tmp/venv",
    "VIRTUAL_ENV": "/tmp/venv",
    "UV_LINK_MODE": "copy",
    "UV_SYSTEM_PYTHON": "0",
    "PATH": os.environ["PATH"] + ":/root/.local/bin",
}
subprocess.run(["just", "sync"], env=env, check=True)
subprocess.run(["just", "data-pull"], env=env, check=True)
subprocess.run(["uv", "run", "--no-sync", "dvc", "pull", "data/weights.dvc"], env=env, check=True)
Path(".env").unlink(missing_ok=True)

In [ ]:
subprocess.run(
    [
        "uv",
        "run",
        "--no-sync",
        "python",
        "-m",
        "vehicle_detection.experiments.compare",
        "--config-name",
        "compare_final_e0",
        "coco_path=data/processed/coco.json",
        "images_dir=data/raw/transset/TRANSSET_v2/images",
        "weights=data/weights/E0-yolo26n-640.pt",
        "manifest_path=splits/v1.csv",
        "split=test",
        "out_dir=runs/final_test_e0",
    ],
    env=env,
    check=True,
)

In [ ]:
print(Path("runs/final_test_e0/comparison.md").read_text())
print(Path("runs/final_test_e0/per_class.md").read_text())